In [ ]:
import sys
import torch
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
        "GB"
    )
else:
    print("No GPU detected. Please enable a GPU runtime.")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU name: Tesla T4
GPU memory: 14.56 GB


In [ ]:
%pip install -q openai-whisper
%pip install -q unsloth
%pip install -q soundfile librosa

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import whisper
import torch
import unsloth
import soundfile as sf
import librosa

print("Whisper version:", whisper.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Unsloth imported successfully")
print("SoundFile imported successfully")
print("Librosa imported successfully")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nAll required libraries imported successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Whisper version: 20250625
PyTorch version: 2.11.0+cu130
CUDA available: True
Unsloth imported successfully
SoundFile imported successfully
Librosa imported successfully
GPU: Tesla T4

All required libraries imported successfully!


In [ ]:
import whisper
import torch

# Select the device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Whisper model...")
print("Device:", device)

# Load Whisper base model
whisper_model = whisper.load_model("base", device=device)

print("\nWhisper model loaded successfully!")
print("Model: Whisper Base")
print("Device:", device)

Loading Whisper model...
Device: cuda


100%|███████████████████████████████████████| 139M/139M [00:04<00:00, 29.2MiB/s]



Whisper model loaded successfully!
Model: Whisper Base
Device: cuda


In [ ]:
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

# Generate a short silent audio file for an initial pipeline check
sample_rate = 16000
duration = 3

audio_data = np.zeros(sample_rate * duration, dtype=np.float32)

sf.write("sample_audio.wav", audio_data, sample_rate)

print("Sample audio file created: sample_audio.wav")
print("Sample rate:", sample_rate, "Hz")
print("Duration:", duration, "seconds")

display(Audio(audio_data, rate=sample_rate))

Sample audio file created: sample_audio.wav
Sample rate: 16000 Hz
Duration: 3 seconds


/usr/local/lib/python3.13/dist-packages/IPython/lib/display.py:174: RuntimeWarning: invalid value encountered in divide
  scaled = data / normalization_factor * 32767
/usr/local/lib/python3.13/dist-packages/IPython/lib/display.py:175: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan


In [ ]:
import whisper
import os

# Check that the recording exists
audio_path = "sample_audio.webm"

if not os.path.exists(audio_path):
    raise FileNotFoundError(
        "Recording not found. Please run the audio recording cell again."
    )

print("Transcribing your recorded question...")

# Convert speech to text using Whisper
transcription_result = whisper_model.transcribe(
    audio_path,
    fp16=torch.cuda.is_available()
)

transcribed_text = transcription_result["text"].strip()

print("\n========== WHISPER TRANSCRIPTION ==========")
print("Recognized question:", transcribed_text)
print("===========================================")

In [ ]:
import torch
from unsloth import FastLanguageModel

model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"

print("Loading the 4-bit quantized LLM...")

llm_model, llm_tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Enable optimized inference
FastLanguageModel.for_inference(llm_model)

print("\nQuantized LLM loaded successfully!")
print("Model:", model_name)
print("4-bit quantization: Enabled")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch

# Ensure the model is ready for inference
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(llm_model)

# Check that Whisper produced a transcription
if not transcribed_text.strip():
    raise ValueError("No text was transcribed. Please check your audio recording.")

print("Question received from Whisper:")
print(transcribed_text)

# Create a prompt for the language model
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful AI assistant. Answer the user's question clearly, "
            "accurately, and in simple language. If the question is medical, "
            "provide general educational information, not a diagnosis."
        )
    },
    {
        "role": "user",
        "content": transcribed_text
    }
]

# Convert messages into the model's expected format
prompt = llm_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize the prompt
inputs = llm_tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda" if torch.cuda.is_available() else "cpu")

print("\nGenerating the answer...")

# Generate response
with torch.inference_mode():
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        use_cache=True
    )

# Extract only newly generated tokens
new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
answer = llm_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("\n========== AI ASSISTANT RESPONSE ==========")
print(answer)
print("===========================================")

In [ ]:
import whisper

result = whisper_model.transcribe(
    audio_file,
    fp16=torch.cuda.is_available()
)

transcribed_text = result["text"].strip()

print("Whisper transcription:")
print(transcribed_text)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"

print("Loading quantized Llama model...")

llm_model, llm_tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

FastLanguageModel.for_inference(llm_model)

print("\nQuantized LLM loaded successfully!")
print("Model:", model_name)
print("4-bit quantization: Enabled")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:

# Prepare the question from Whisper
question = transcribed_text

print("Question detected by Whisper:")
print(question)

# Create a prompt for the LLM
prompt = f"""You are a helpful assistant.
Answer the user's question clearly and accurately.
If the question is medical, provide general educational information
and do not claim to diagnose or replace a healthcare professional.

User question: {question}

Answer:"""

# Tokenize the prompt
inputs = llm_tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda" if torch.cuda.is_available() else "cpu")

# Generate the answer
with torch.no_grad():
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        use_cache=True,
    )

# Extract only the generated answer
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
answer = llm_tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print("\nLLM Answer:")
print(answer)

In [ ]:

import torch
import whisper

def speech_to_reasoning(audio_path, max_new_tokens=200):
    print("=" * 60)
    print("SPEECH-TO-REASONING PIPELINE")
    print("=" * 60)

    # Step 1: Transcribe audio using Whisper
    print("\n1. Transcribing audio with Whisper...")

    transcription_result = whisper_model.transcribe(
        audio_path,
        fp16=torch.cuda.is_available()
    )

    question = transcription_result["text"].strip()

    print("Transcribed Question:", question)

    if not question:
        print("No speech detected. Please provide a clearer recording.")
        return None

    # Step 2: Prepare the LLM prompt
    print("\n2. Sending transcription to the quantized LLM...")

    prompt = f"""You are a helpful assistant.
Answer the following question clearly and accurately.
For medical questions, provide general educational information only.

Question: {question}

Answer:"""

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    # Step 3: Generate the answer
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    print("\n3. Generated Answer:")
    print(answer)

    # Step 4: Show GPU memory usage
    if torch.cuda.is_available():
        memory_used = torch.cuda.memory_allocated() / (1024**3)
        print(f"\nGPU memory currently allocated: {memory_used:.2f} GB")

    print("\nPipeline completed!")
    print("=" * 60)

    return {
        "transcription": question,
        "answer": answer
    }

print("Speech-to-reasoning function created successfully!")

In [ ]:
result = speech_to_reasoning(audio_file)

if result:
    print("\nFINAL RESULT")
    print("Question:", result["transcription"])
    print("Answer:", result["answer"])

In [ ]:
%pip install -q gTTS

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

# Example question for our speech-to-reasoning pipeline
sample_question = "What are the common symptoms of anemia?"

# Generate AI speech
tts = gTTS(text=sample_question, lang="en")
tts.save("ai_generated_question.mp3")

print("AI-generated voice question:")
print(sample_question)

# Play the generated audio inside Colab
display(Audio("ai_generated_question.mp3"))

In [ ]:
result = speech_to_reasoning("ai_generated_question.mp3")

print("\nFINAL PIPELINE OUTPUT")
print("Transcribed question:", result["transcription"])
print("AI assistant answer:", result["answer"])

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

sample_question_2 = "What are some common ways to maintain a healthy lifestyle?"

tts = gTTS(text=sample_question_2, lang="en")
tts.save("ai_generated_question_2.mp3")

print("Generated question:", sample_question_2)
display(Audio("ai_generated_question_2.mp3"))

In [ ]:
result_2 = speech_to_reasoning("ai_generated_question_2.mp3")

if result_2:
    print("\nFINAL PIPELINE OUTPUT")
    print("Transcribed question:", result_2["transcription"])
    print("AI assistant answer:", result_2["answer"])

In [ ]:
import time
import torch

start_time = time.time()

result_2 = speech_to_reasoning("ai_generated_question_2.mp3")

end_time = time.time()
total_time = end_time - start_time

print("\n" + "=" * 50)
print("PERFORMANCE EVALUATION")
print("=" * 50)

if result_2:
    print("Input Audio:", "ai_generated_question_2.mp3")
    print("Transcription:", result_2["transcription"])
    print("Generated Answer:", result_2["answer"])
    print(f"Total Pipeline Time: {total_time:.2f} seconds")

    if torch.cuda.is_available():
        print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
        print(f"GPU Memory Reserved: {torch.cuda.memory_reserved() / (1024**3):.2f} GB")

In [ ]:
import json
import os
from datetime import datetime

# Collect results from the two voice examples
results_summary = {
    "project_title": "Speech-to-Reasoning Pipeline with Whisper and Quantized LLM",
    "task": "Arch Technologies Generative AI Internship - Task 4",
    "model": "Llama 3.2 1B Instruct (4-bit quantized)",
    "speech_model": "OpenAI Whisper Base",
    "examples": []
}

# First example
if "result" in globals() and result:
    results_summary["examples"].append({
        "audio_file": "ai_generated_question.mp3",
        "transcription": result["transcription"],
        "generated_answer": result["answer"]
    })

# Second example
if "result_2" in globals() and result_2:
    results_summary["examples"].append({
        "audio_file": "ai_generated_question_2.mp3",
        "transcription": result_2["transcription"],
        "generated_answer": result_2["answer"]
    })

results_summary["evaluation_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with open("task4_results.json", "w", encoding="utf-8") as f:
    json.dump(results_summary, f, indent=4, ensure_ascii=False)

print("Results saved successfully!")
print("Number of examples saved:", len(results_summary["examples"]))
print(json.dumps(results_summary, indent=2, ensure_ascii=False))